# 1. Data: ABCD as typed questions

ABCD (ASAPP, MIT licence) has about 10k customer↔agent chats with official train/dev/test splits. Each chat becomes one `state` and three typed questions with gold labels:

| Question | Type | Options | What it tests |
|---|---|---|---|
| `flow` | `choice` | 10 | Generic intents: out-of-the-box engines should do well |
| `subflow` | `choice` | 55 | ABCD's own fine-grained labels: where LoRA should help |
| `is_product_return` | `noul` | yes/no | Calibration |

Agent tool actions are dropped from the `state`, because they give the answer away. Names and emails in ABCD are synthetic.

In [1]:
from bench.data import abcd

questions = abcd.questions()
splits = {s: abcd.load(s) for s in ['train', 'dev', 'test']}
{s: len(df) for s, df in splits.items()}

{'train': 8034, 'dev': 1004, 'test': 1004}

## Label balance (test)

In [2]:
test = splits['test']
display(test['flow'].value_counts())
print(f"is_product_return: {test['is_product_return'].mean():.0%} yes")
print(f"subflows in test: {test['subflow'].nunique()}")

flow
purchase_dispute        111
storewide_query         107
shipping_issue          106
single_item_query       106
product_defect          105
order_issue             105
troubleshoot_site       103
account_access           99
subscription_inquiry     94
manage_account           68
Name: count, dtype: int64

is_product_return: 10% yes
subflows in test: 96


## Example `state`

In [3]:
print(test.loc[0, 'state'])
test.loc[0, ['flow', 'subflow', 'is_product_return']]

Agent: Hello. How can i help you today?
Customer: Hi.  My name is Chloe Zhang.  I am curious as to when my promo code expires.
Customer: Would you be able to tell me?
Agent: Yes let me look into this im sure we can find a solution
Agent: Okay I've looked into this for you
Agent: Promo codes will expire 7 days after they are issued.
Customer: Ok.
Agent: Is there anything else I can help you with?
Customer: No that is all for today.  Thank you.


flow                          storewide_query
subflow              storewide_query.timing_4
is_product_return                       False
Name: 0, dtype: object

## State length

This drives cost and latency. At roughly 4 characters per token, a typical chat is about 250 tokens.

In [4]:
test['state'].str.len().describe(percentiles=[0.5, 0.95]).round(0)

count    1004.0
mean      935.0
std       333.0
min       316.0
50%       885.0
95%      1545.0
max      2538.0
Name: state, dtype: float64

## Frozen test sample

The API engines cost money and time per call, so they all use the same fixed 300-conversation sample of `test` (seed 0). The full test split is kept for decider runs.

In [5]:
bench_test = abcd.load('test', limit=300, seed=0)
bench_test['flow'].value_counts()

flow
purchase_dispute        43
order_issue             37
subscription_inquiry    35
troubleshoot_site       32
single_item_query       30
account_access          30
shipping_issue          29
product_defect          27
storewide_query         25
manage_account          12
Name: count, dtype: int64